In [ ]:
# Setup: load config, create paths, and initialize basic dependencies.
# This cell keeps defaults small (7 days, small subreddit/query set).
# It also prepares a dry-run fallback if API credentials are missing.
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path

import nltk
import pandas as pd
from dotenv import load_dotenv

from src.reddit_client import RedditClient
from src.sentiment import VaderSentiment
from src.text_utils import assemble_doc_text
from src.themes import detect_themes, THEME_KEYWORDS
from src.viz import save_negative_themes, save_sentiment_trend

load_dotenv()

Path("outputs").mkdir(exist_ok=True)
Path("data").mkdir(exist_ok=True)
Path("cache").mkdir(exist_ok=True)

try:
    nltk.data.find("sentiment/vader_lexicon.zip")
except LookupError:
    nltk.download("vader_lexicon")

SUBREDDITS = ["instacart", "InstacartShoppers", "Frugal"]
QUERY_TERMS = [
    "instacart",
    "instacart fees",
    "instacart refunds",
    "instacart substitutions",
    "instacart missing wrong items",
    "instacart late canceled",
    "instacart tipping support",
]
POST_LIMIT = 40
DAYS_BACK = 7
CUTOFF_TS = int((datetime.now(timezone.utc) - timedelta(days=DAYS_BACK)).timestamp())

client_id = os.getenv("REDDIT_CLIENT_ID")
client_secret = os.getenv("REDDIT_CLIENT_SECRET")
user_agent = os.getenv("REDDIT_USER_AGENT")
has_creds = bool(client_id and client_secret and user_agent)
USE_SAMPLE_DATA = os.getenv("USE_SAMPLE_DATA", "1") == "1"
print(f"Credentials available: {has_creds}")
print(f"Using sample synthetic data: {USE_SAMPLE_DATA}")

In [ ]:
# Pull posts from Reddit API and store in SQLite when credentials exist.
# If creds are missing, this cell tries a local sample CSV fallback.
# The schema avoids usernames and keeps only post-level fields.
all_posts = []
sample_path = Path("data/sample_input.csv")

if USE_SAMPLE_DATA:
    if not sample_path.exists():
        raise RuntimeError("Sample mode is enabled but data/sample_input.csv was not found.")
    posts_df = pd.read_csv(sample_path)
elif has_creds:
    client = RedditClient(
        client_id=client_id,
        client_secret=client_secret,
        user_agent=user_agent,
        cache_dir="cache",
        db_path="data/reddit_pulse.db",
    )
    for subreddit in SUBREDDITS:
        for query in QUERY_TERMS:
            posts = client.fetch_submissions(subreddit=subreddit, query=query, limit=POST_LIMIT)
            posts = [p for p in posts if p["created_utc"] >= CUTOFF_TS]
            all_posts.extend(posts)
    dedup = {p["post_id"]: p for p in all_posts if p.get("post_id")}
    posts_df = pd.DataFrame(dedup.values())
    client.upsert_posts(posts_df.to_dict("records"))
else:
    raise RuntimeError("Missing Reddit credentials. Add .env and set USE_SAMPLE_DATA=0, or use bundled sample mode.")

print(f"Posts captured: {len(posts_df)}")
posts_df.head(3)

In [ ]:
# Pull a small comment sample per post and store to SQLite.
# This limits depth/volume to stay API-safe and analysis-friendly.
# In dry-run mode with no credentials, comments default to empty.
comments_df = pd.DataFrame(columns=["comment_id", "post_id", "subreddit", "created_utc", "created_dt", "body", "permalink", "score", "retrieved_utc"])

if has_creds and not USE_SAMPLE_DATA:
    comment_rows = []
    for _, row in posts_df.iterrows():
        subset = client.fetch_comments(row["subreddit"], row["post_id"], limit=20, depth=2)
        comment_rows.extend(subset)
    if comment_rows:
        dedup_comments = {c["comment_id"]: c for c in comment_rows if c.get("comment_id")}
        comments_df = pd.DataFrame(dedup_comments.values())
        client.upsert_comments(comments_df.to_dict("records"))

print(f"Comments captured: {len(comments_df)}")
comments_df.head(3)

In [ ]:
# Build post documents from title/body/top comments and run VADER sentiment.
# Sentiment labels use simple threshold rules for product-level direction.
# Output is one analyzed row per post.
sentiment = VaderSentiment()

if not comments_df.empty:
    grouped_comments = comments_df.groupby("post_id")["body"].apply(list).to_dict()
else:
    grouped_comments = {}

analysis_rows = []
for _, row in posts_df.iterrows():
    doc_text = assemble_doc_text(row.get("title", ""), row.get("body", ""), grouped_comments.get(row["post_id"], []))
    scores = sentiment.score(doc_text)
    analysis_rows.append(
        {
            **row.to_dict(),
            "doc_text": doc_text,
            "compound": scores["compound"],
            "sentiment": sentiment.label(scores["compound"]),
            "day": datetime.utcfromtimestamp(int(row["created_utc"])).strftime("%Y-%m-%d"),
        }
    )

analysis_df = pd.DataFrame(analysis_rows)
analysis_df[["post_id", "subreddit", "sentiment", "compound"]].head(5)

In [ ]:
# Tag each post with keyword themes and build core metric tables.
# Themes are multi-label and focused on practical product pain points.
# These tables feed both the charts and the memo.
analysis_df["themes"] = analysis_df["doc_text"].apply(detect_themes)
analysis_df["is_negative"] = analysis_df["sentiment"].eq("negative")

volume_by_subreddit = analysis_df.groupby("subreddit").agg(posts=("post_id", "count")).reset_index()
comment_volume_by_subreddit = comments_df.groupby("subreddit").agg(comments=("comment_id", "count")).reset_index() if not comments_df.empty else pd.DataFrame(columns=["subreddit", "comments"])

daily = analysis_df.groupby("day").agg(post_count=("post_id", "count"), negative_count=("is_negative", "sum")).reset_index()
daily["negative_share"] = (daily["negative_count"] / daily["post_count"] * 100).round(1)

sentiment_dist = analysis_df["sentiment"].value_counts(normalize=True).mul(100).round(1).rename_axis("sentiment").reset_index(name="pct")
sentiment_by_subreddit = analysis_df.pivot_table(index="subreddit", columns="sentiment", values="post_id", aggfunc="count", fill_value=0)

negative_posts = analysis_df[analysis_df["is_negative"]].copy()
negative_theme_rows = []
for _, r in negative_posts.iterrows():
    for t in r["themes"]:
        negative_theme_rows.append({"post_id": r["post_id"], "subreddit": r["subreddit"], "theme": t})
negative_theme_df = pd.DataFrame(negative_theme_rows)

if negative_theme_df.empty:
    theme_counts = pd.DataFrame({"theme": list(THEME_KEYWORDS.keys()), "count": [0] * len(THEME_KEYWORDS)})
else:
    theme_counts = negative_theme_df.groupby("theme").agg(count=("post_id", "nunique")).reset_index().sort_values("count", ascending=False)

cross_subreddit_repeat = pd.DataFrame(columns=["theme", "subreddit_count"])
if not negative_theme_df.empty:
    cross_subreddit_repeat = negative_theme_df.groupby("theme")["subreddit"].nunique().reset_index(name="subreddit_count")
    cross_subreddit_repeat = cross_subreddit_repeat[cross_subreddit_repeat["subreddit_count"] >= 2].sort_values("subreddit_count", ascending=False)

print(volume_by_subreddit)
print(sentiment_dist)

In [ ]:
# Render and save chart outputs for trend and top negative themes.
# These figures are designed for quick scanning in a product memo.
# Only aggregated outputs are saved.
save_sentiment_trend(daily.sort_values("day"), "outputs/sentiment_trend.png")
save_negative_themes(theme_counts.head(6), "outputs/negative_themes.png")

print("Saved charts:")
print("- outputs/sentiment_trend.png")
print("- outputs/negative_themes.png")

In [ ]:
# Generate a short APM-style product memo from aggregated metrics.
# The memo emphasizes what happened, why it matters, and next bets.
# No raw post/comment text is written to outputs.
start_day = analysis_df["day"].min() if not analysis_df.empty else "N/A"
end_day = analysis_df["day"].max() if not analysis_df.empty else "N/A"

top_theme_rows = theme_counts.head(3).to_dict("records")
exec_summary = [
    f"Sampled {len(analysis_df)} posts and {len(comments_df)} comments across {analysis_df['subreddit'].nunique()} subreddits.",
    f"Overall sentiment split: " + ", ".join([f"{r['sentiment']} {r['pct']}%" for _, r in sentiment_dist.iterrows()]),
    f"Highest negative-share day: {daily.sort_values('negative_share', ascending=False).iloc[0]['day']} ({daily.sort_values('negative_share', ascending=False).iloc[0]['negative_share']}%)." if not daily.empty else "No daily trend available.",
]

observed_lines = []
for r in top_theme_rows:
    share = 0
    if len(negative_posts) > 0:
        share = round(r["count"] / len(negative_posts) * 100, 1)
    observed_lines.append(f"- **{r['theme']}** appeared in {r['count']} negative posts ({share}% of negative posts).")

repeat_lines = "\n".join([f"- {r.theme}: present in {r.subreddit_count} subreddits" for r in cross_subreddit_repeat.itertuples()]) or "- No theme appeared in 2+ subreddits in this run."

memo = f"""# Instacart Reddit Pulse Memo

**Date range:** {start_day} to {end_day}  
**Data source:** Reddit public posts/comments via official OAuth API (small, directional sample).

## Executive summary
- {exec_summary[0]}
- {exec_summary[1]}
- {exec_summary[2]}

## What we observed
{chr(10).join(observed_lines) if observed_lines else '- No major negative themes found.'}

### Cross-subreddit repeat pain
{repeat_lines}

## Why it matters
- **Consumer side:** Persistent issues in fees, substitutions, and order accuracy can reduce repeat order intent.
- **Shopper side:** Delivery reliability and tipping/support friction can lower shopper supply quality and fill rate.
- **Marketplace risk:** When both sides report friction, service trust can decline faster than isolated incident metrics suggest.

## Recommendations
1. **Pricing clarity bet:** Add pre-checkout fee transparency and compare complaint-rate changes for fee-related sessions.
2. **Substitution confidence bet:** Improve substitution prompts with explicit item-quality preferences and track substitution acceptance + refund rates.
3. **Support recovery bet:** Add faster in-flow support for missing/wrong items with instant credit guardrails and monitor CSAT + repeat purchase.

## How to validate
1. **Experiment:** Fee transparency variant in checkout. **Success metric:** reduction in fee-related negative-theme share by 20% over 2 weeks.
2. **Experiment:** Substitution UX nudge for top categories. **Success metric:** +5% accepted substitutions, lower complaint volume.
3. **Experiment:** High-priority support lane for order-accuracy incidents. **Success metric:** faster resolution time and lower 7-day churn proxy.

## Limitations
- Small sample and subreddit composition bias.
- VADER + keyword rules can miss sarcasm and context.
- Results are directional, not causal.

## Ethics/compliance
- Aggregated outputs only; no raw post/comment text published in repo outputs.
- Local cached/raw data should be removed quickly using `python scripts/cleanup_local_data.py` (within 48 hours).
"""

Path("outputs/product_memo.md").write_text(memo)
print("Saved memo: outputs/product_memo.md")

In [ ]:
# Final reminder: keep runs lightweight and clean local data after use.
print("Next steps:")
print("1) Review outputs/product_memo.md")
print("2) Share charts with product stakeholders")
print("3) Run cleanup script within 48 hours: python scripts/cleanup_local_data.py")